# Phase 0 — SU(2) Glueball Spectrum on GPU

**Lattice:** 12³ × 24 | **Group:** SU(2) | **β:** 2.5 | **Backend:** CuPy (T4 GPU)

**Goal:** Validate plaquette (~0.65) and extract 0⁺⁺ glueball effective mass.

**Runtime estimate:** ~2–4 hours on T4 (free Colab)

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q cupy-cuda12x numpy matplotlib

In [ ]:
# Clone repo
!git clone https://github.com/Jefemaestro33/yang-mills-lattice.git
%cd yang-mills-lattice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from su2_lattice import (
    set_backend, GaugeField, glueball_correlator, effective_mass, xp
)

set_backend(use_gpu=True)

## 1. Parameters

In [ ]:
# Phase 0 parameters
Nt, Ns = 24, 12
dims = (Nt, Ns, Ns, Ns)
beta = 2.5

n_therm = 200       # thermalization sweeps
n_cfg   = 500       # configurations to generate
n_skip  = 50        # sweeps between measurements
n_smear = 30        # APE smearing iterations
alpha   = 0.5       # smearing parameter

print(f"Lattice: {Ns}³ × {Nt}")
print(f"β = {beta}")
print(f"Configs: {n_cfg}, skip: {n_skip}, smearing: {n_smear} iters (α={alpha})")
print(f"Total sweeps: {n_therm + n_cfg * n_skip}")

## 2. Thermalization

In [ ]:
gf = GaugeField(dims, beta=beta, start='hot')

# Time one sweep
t0 = time.time()
gf.sweep()
dt = time.time() - t0
total_est = (n_therm + n_cfg * n_skip) * dt / 3600
print(f"1 sweep = {dt:.3f}s → estimated total: {total_est:.1f} hours\n")

# Thermalize
plaq_therm = []
t0 = time.time()
for i in range(1, n_therm + 1):
    gf.sweep()
    if i % 20 == 0:
        p = gf.plaquette()
        plaq_therm.append((i, p))
        if i % 50 == 0:
            print(f"  sweep {i:4d}/{n_therm}  ⟨P⟩ = {p:.6f}  [{time.time()-t0:.0f}s]")

print(f"\nThermalization done in {time.time()-t0:.0f}s")

## 3. Measurement

In [ ]:
plaq_history = []
O_smeared = []
O_unsmeared = []

t0 = time.time()
for icfg in range(n_cfg):
    for _ in range(n_skip):
        gf.sweep()
    
    p = gf.plaquette()
    plaq_history.append(p)
    
    # Smeared operator (glueball signal)
    O_sm = gf.smeared_spatial_plaquette_timeslice(n_smear=n_smear, alpha=alpha)
    # Unsmeared operator (comparison)
    O_un = gf.spatial_plaquette_timeslice()
    
    # CuPy → NumPy
    import cupy
    if isinstance(O_sm, cupy.ndarray):
        O_sm = O_sm.get()
        O_un = O_un.get()
    
    O_smeared.append(O_sm)
    O_unsmeared.append(O_un)
    
    if (icfg + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (icfg + 1) * (n_cfg - icfg - 1)
        print(f"  cfg {icfg+1:4d}/{n_cfg}  ⟨P⟩={p:.5f}  "
              f"[{elapsed/60:.1f}min elapsed, ~{eta/60:.1f}min remaining]")

# Save checkpoint
plaq_history = np.array(plaq_history)
O_smeared = np.array(O_smeared)
O_unsmeared = np.array(O_unsmeared)

!mkdir -p results
np.savez('results/phase0_gpu.npz',
    dims=np.array(dims), beta=beta,
    plaquette_history=plaq_history,
    O_smeared=O_smeared,
    O_unsmeared=O_unsmeared,
    n_smear=n_smear, alpha=alpha)

dt_total = time.time() - t0
print(f"\nDone: {n_cfg} configs in {dt_total/3600:.2f} hours")
print(f"Saved to results/phase0_gpu.npz")

## 4. Analysis

In [ ]:
# Plaquette
print(f"⟨P⟩ = {np.mean(plaq_history):.6f} ± {np.std(plaq_history)/np.sqrt(len(plaq_history)):.6f}")
print(f"Expected: ~0.652")
print()

In [ ]:
# Correlators
C_sm, Ce_sm = glueball_correlator(O_smeared)
C_un, Ce_un = glueball_correlator(O_unsmeared)
m_sm, dm_sm = effective_mass(C_sm, Ce_sm)
m_un, dm_un = effective_mass(C_un, Ce_un)

print("=== SMEARED ===")
print(f"{'τ':>3s}  {'C(τ)':>12s} {'±':>1s} {'err':>10s}  {'m_eff':>8s} {'±':>1s} {'err':>8s}  {'S/N':>5s}")
for tau in range(Nt // 2 + 1):
    sn = abs(C_sm[tau]) / max(Ce_sm[tau], 1e-30)
    m_str = ""
    if tau < len(m_sm) and not np.isnan(m_sm[tau]):
        m_str = f"{m_sm[tau]:8.4f} ± {dm_sm[tau]:8.4f}"
    print(f"{tau:3d}  {C_sm[tau]:12.2f} ± {Ce_sm[tau]:10.2f}  {m_str:>20s}  {sn:5.1f}")

print("\n=== UNSMEARED ===")
for tau in range(min(6, Nt // 2 + 1)):
    sn = abs(C_un[tau]) / max(Ce_un[tau], 1e-30)
    print(f"  τ={tau}: C={C_un[tau]:10.2f} ± {Ce_un[tau]:8.2f}  S/N={sn:.1f}")

In [ ]:
# Plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plaquette history
ax = axes[0]
ax.plot(plaq_history, 'o-', ms=1.5, lw=0.3)
ax.axhline(np.mean(plaq_history), color='r', ls='--',
           label=f'⟨P⟩ = {np.mean(plaq_history):.5f}')
ax.set_xlabel('Configuration')
ax.set_ylabel('⟨P⟩')
ax.set_title('Plaquette history')
ax.legend()

# Correlator (log scale)
ax = axes[1]
tau = np.arange(len(C_sm))
pos = C_sm > 0
ax.errorbar(tau[pos], C_sm[pos], yerr=Ce_sm[pos],
            fmt='o-', ms=4, capsize=2, label='smeared')
pos_u = C_un > 0
ax.errorbar(tau[pos_u] + 0.15, C_un[pos_u], yerr=Ce_un[pos_u],
            fmt='s-', ms=3, capsize=2, alpha=0.5, label='unsmeared')
ax.set_yscale('log')
ax.set_xlabel('τ')
ax.set_ylabel('C(τ)')
ax.set_title('0⁺⁺ Correlator')
ax.set_xlim(-0.5, Nt // 2 + 1)
ax.legend()

# Effective mass
ax = axes[2]
tau_m = np.arange(len(m_sm))
good = ~np.isnan(m_sm)
if dm_sm is not None:
    good &= ~np.isnan(dm_sm)
    ax.errorbar(tau_m[good], m_sm[good], yerr=dm_sm[good],
                fmt='o-', ms=5, capsize=3, label='smeared')
good_u = ~np.isnan(m_un)
if dm_un is not None:
    good_u &= ~np.isnan(dm_un)
    ax.errorbar(tau_m[good_u] + 0.1, m_un[good_u], yerr=dm_un[good_u],
                fmt='s-', ms=4, capsize=2, alpha=0.5, label='unsmeared')
ax.set_xlabel('τ')
ax.set_ylabel('m_eff(τ) [lattice units]')
ax.set_title('Effective mass')
ax.set_xlim(-0.5, 8)
ax.legend()

fig.suptitle(f'Phase 0 — SU(2) {Ns}³×{Nt}  β={beta}  {n_cfg} cfgs  smear={n_smear}',
             fontsize=13)
fig.tight_layout()
fig.savefig('results/phase0_plots.png', dpi=150)
plt.show()
print('Saved to results/phase0_plots.png')

## 5. Checkpoint: Pass/Fail

**Plaquette:** ⟨P⟩ ≈ 0.652 ± small → ✓ or ✗

**Correlator:** Exponential decay visible for τ = 0, 1, 2 → ✓ or ✗

**Effective mass:** Plateau visible at τ = 1–3 → estimate m_eff in lattice units

If both pass → proceed to Phase 1 ($15–20, A100 spot instance)

In [ ]:
# Download results
from google.colab import files
files.download('results/phase0_gpu.npz')
files.download('results/phase0_plots.png')